# C5 – Numerische Validierung der MPI-Wärmesimulation

**Ziel:** Prüfen, ob die Berechnung mit **einem** und **vier MPI-Prozessen** an jedem Gitterpunkt dieselbe Temperatur liefert. Das ist ein **Korrektheitsnachweis**, keine Leistungsmessung.

**Vorgaben aus dem Arbeitsplan (C5):** 512 × 512 Gitterpunkte, 200 Zeitschritte, Fourier-Zahl `r = 0,2`. Die mit `heat_bild.py` auf dem Ubuntu-Manager erzeugten Dateien werden **nur eingelesen**. Das Notebook startet **keine MPI-Prozesse** und verändert keine bestehenden Simulationsprogramme.

**Ausführung:** Dieses Notebook unter `/home/itmvs/waerme/niklas/` auf dem Ubuntu-Manager öffnen und alle Codezellen ausführen. Benötigt wird lediglich NumPy (in `~/venv` vorhanden).

In [1]:
from pathlib import Path
import numpy as np
import csv

# Die originalen C5-Simulationsdaten liegen auf dem Ubuntu-Manager.
BASIS = Path("/home/itmvs/waerme/niklas")
DATEI_P1 = BASIS / "gitter_N512_P1_S200.npy"
DATEI_P4 = BASIS / "gitter_N512_P4_S200.npy"

for datei in (DATEI_P1, DATEI_P4):
    if not datei.is_file():
        raise FileNotFoundError(
            f"C5-Eingabedatei fehlt: {datei}\n"
            "Bitte zuerst die C5-Rechnungen auf dem Ubuntu-Manager durchführen."
        )

A = np.load(DATEI_P1, allow_pickle=False)
B = np.load(DATEI_P4, allow_pickle=False)

# Ein Vergleich ergibt nur Sinn, wenn beide Gitter dieselbe Form besitzen.
if A.shape != (512, 512) or B.shape != (512, 512):
    raise ValueError(f"Erwartet: (512, 512). Gefunden: P1={A.shape}, P4={B.shape}")
if not (np.isfinite(A).all() and np.isfinite(B).all()):
    raise ValueError("Mindestens eine Datei enthält NaN oder unendliche Werte.")

print("Beide Originaldateien erfolgreich geladen.")
print(f"P1: {A.shape}, P4: {B.shape}")

Beide Originaldateien erfolgreich geladen.
P1: (512, 512), P4: (512, 512)


## Prüfschritte

- **Größte Abweichung:** größte Temperaturdifferenz zwischen denselben Zellen der beiden Ergebnisse. Erwartung aus dem Arbeitsplan: `0,0`.
- **Temperaturgrenzen:** bei stabiler FTCS-Rechnung liegen alle Temperaturen zwischen `0` und `100`.
- **Links-rechts-Symmetrie:** Vergleiche die 4-Prozess-Platte mit ihrem gespiegelten Bild. Eine winzige Abweichung um `10⁻¹⁴` ist durch Rundung möglich.
- **Randwerte:** Oben ist die gesamte Zeile `100`; unten und an den Seiten unterhalb der oberen Ecken `0`.

Die gemessenen Werte werden **aus den eingelesenen Arrays berechnet**, nicht aus bisherigen Ergebnissen übernommen.

In [2]:
# Genau dieselbe Berechnung, nur mit unterschiedlich vielen MPI-Prozessen:
abweichung = float(np.max(np.abs(A - B)))
gleich_bis_aufs_bit = bool(np.array_equal(A, B))
min_p1, max_p1 = float(A.min()), float(A.max())
min_p4, max_p4 = float(B.min()), float(B.max())
symmetrie = float(np.max(np.abs(B - B[:, ::-1])))

# Sehr kleine Rundungsfehler bei Symmetrie sind erlaubt.
toleranz = 1e-10
oben_korrekt = bool(np.all(B[0, :] == 100.0))
unten_korrekt = bool(np.all(B[-1, :] == 0.0))
seiten_korrekt = bool(np.all(B[1:, 0] == 0.0) and np.all(B[1:, -1] == 0.0))

pruefungen = {
    "Gitter beider Ergebnisse 512 × 512": A.shape == B.shape == (512, 512),
    "Numerische Ergebnisse exakt gleich": gleich_bis_aufs_bit,
    "Temperaturwerte zwischen 0 und 100": min_p1 >= -toleranz and max_p1 <= 100 + toleranz and min_p4 >= -toleranz and max_p4 <= 100 + toleranz,
    "Links-rechts-Symmetrie bis auf Rundung": symmetrie < toleranz,
    "Oberer Rand dauerhaft 100": oben_korrekt,
    "Unterer und seitliche Ränder 0": unten_korrekt and seiten_korrekt,
}

werte = [
    ("Gittergröße", f"{A.shape[0]} × {A.shape[1]}"),
    ("Zeitschritte", "200"),
    ("Prozesse", "1 vs. 4"),
    ("Größte Temperaturabweichung", f"{abweichung:.17g}"),
    ("Kleinste Temperatur P1", f"{min_p1:.17g}"),
    ("Größte Temperatur P1", f"{max_p1:.17g}"),
    ("Kleinste Temperatur P4", f"{min_p4:.17g}"),
    ("Größte Temperatur P4", f"{max_p4:.17g}"),
    ("Links-rechts-Symmetrie P4", f"{symmetrie:.17g}"),
]

print("C5 – Validierungsergebnisse")
print("=" * 64)
for name, wert in werte:
    print(f"{name:<36} {wert}")
print("\nPrüfungen")
for name, korrekt in pruefungen.items():
    print(f"{'OK' if korrekt else 'NICHT BESTANDEN'}: {name}")

if all(pruefungen.values()):
    print("\nErgebnis: Alle C5-Prüfungen bestanden.")
else:
    print("\nErgebnis: Mindestens eine Prüfung ist fehlgeschlagen. Ergebnisse prüfen!")

C5 – Validierungsergebnisse
Gittergröße                          512 × 512
Zeitschritte                         200
Prozesse                             1 vs. 4
Größte Temperaturabweichung          0
Kleinste Temperatur P1               0
Größte Temperatur P1                 100
Kleinste Temperatur P4               0
Größte Temperatur P4                 100
Links-rechts-Symmetrie P4            2.8421709430404007e-14

Prüfungen
OK: Gitter beider Ergebnisse 512 × 512
OK: Numerische Ergebnisse exakt gleich
OK: Temperaturwerte zwischen 0 und 100
OK: Links-rechts-Symmetrie bis auf Rundung
OK: Oberer Rand dauerhaft 100
OK: Unterer und seitliche Ränder 0

Ergebnis: Alle C5-Prüfungen bestanden.


## Ergebnisse für den Bericht speichern

Die nächste Zelle speichert die **neu berechneten** Kennzahlen als kleine CSV-Datei in deinem eigenen Verzeichnis. Die `.npy`-Originaldateien werden **nicht verändert**. Die CSV eignet sich zur Übernahme in euren Projektbericht.

In [3]:
# Optional: auf False setzen, wenn keine CSV-Datei erstellt werden soll.
SPEICHERE_CSV = True

if SPEICHERE_CSV:
    zieldatei = BASIS / "validierung_c5_ergebnisse.csv"
    with zieldatei.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f, delimiter=";")
        writer.writerow(["C5 – Validierung im Kleinen", "Wert"])
        writer.writerows(werte)
        writer.writerow([])
        writer.writerow(["Prüfung", "Bestanden"])
        writer.writerows((name, "Ja" if korrekt else "Nein") for name, korrekt in pruefungen.items())
    print(f"CSV gespeichert: {zieldatei}")

CSV gespeichert: /home/itmvs/waerme/niklas/validierung_c5_ergebnisse.csv


## Einordnung für den wissenschaftlichen Bericht

**Was C5 zeigt:** Wenn die maximale Differenz `0,0` beträgt, stimmen **alle 512 × 512 Temperaturwerte** nach 200 Zeitschritten bei einem und vier MPI-Prozessen exakt überein. Das belegt die numerische Übereinstimmung **für diese Testkonfiguration**. Die erwartete Symmetrie und die Randtemperaturen dienen als weitere Plausibilitätsprüfungen.

**Was C5 noch nicht beweist:** Die vier Prozesse laufen auf dem **Manager** – es ist damit noch kein Nachweis für die Verteilung auf vier **physische Rechner**. Der folgt bei C6. Die Laufzeit- und Speedup-Analyse gehört zu Person A und erfordert wiederholte, getrennte Messreihen.

**Bisher gemeldete C5-Ergebnisse (zur Kontrolle, nicht als feste Berechnungswerte):** maximale Abweichung `0,0`, Temperaturbereich `0–100`, Links-rechts-Symmetrieabweichung `2,842170943040401 × 10⁻¹⁴`.

**Quelle:** *Arbeitsplan Team ITM (5 Personen)*, Abschnitt C5 (S. 11).